# Heat Hack 2026: The Heatwave Diagnostics Package (HDP)

`Developer: Cameron Cummins` </br>
`Title: Computational Geoscientist` </br>
`Affiliation: UT Austin`

*This notebooks is designed as a tutorial introduction to the HDP using NCAR's Casper HPC JupyterHub interface and CESM2 Large Ensemble temperature datasets.*

**Make sure to select the `NPL 2026a` Python environment before running this notebook by using the dropdown menu in the upper right corner (next to the "Notebook" and "Debug" bug button).**

## 1. Setup

### 1.1 Verify your version

Try running the code cell below. It most likely won't work unless you have already completed step `1.2`. This utility function confirms the version of HDP you are using:

In [ ]:
import hdp.utils

hdp.utils.get_version()

### 1.2 Install the HDP
If you got a `ModuleNotFoundError`, don't panic! This is expected because you need to install the package before you can use it. Run the code below to install the HDP, restart your kernel (by clicking the circular arrow button in the toolbar), and re-run the cell above to verify your version.

In [ ]:
!pip install -U hdp-python

## 2. Explore the Data

Let's explore some input data. For this example, we will explore two datasets from the [CESM2 Large Ensemble (LE)](https://www.cesm.ucar.edu/community-projects/lens2):

1. CESM2 LE Historical Daily TREFHTMN (daily minimum temperature at reference height) from 1961 to 1990
2. CESM2 LE SSP3-7.0 Daily TREFHTMN from 2015 to 2050

Both datasets are located here on Casper:

```bash
/glade/campaign/collections/gdex/data/d651056/CESM2-LE/atm/proc/tseries/day_1/TREFHTMN/
```

Each of these datasets contain many members, but for simplicity we will work with a single member of the ensemble, namely `*LE2-1251.001*`.

Use the `xarray` package to load the input data. The HDP is an extension of the `xarray` Dataet/DataArray framework, so if you are familiar with `xarray` you will find the HDP workflow to be very similar.

In [ ]:
import xarray as xr

head_input_directory = "/glade/campaign/collections/gdex/data/d651056/CESM2-LE/atm/proc/tseries/day_1/TREFHTMN/"

trefhtmn_hist = xr.open_mfdataset(
        f"{head_input_directory}*BHIST*LE2-1251.001*.nc",
        data_vars='all'
    ).sel(time=slice("1961-01-01", "1990-12-31"))["TREFHTMN"]

trefhtmn_ssp370 = xr.open_mfdataset(
        f"{head_input_directory}*BSSP370*LE2-1251.001*.nc",
        data_vars='all'
    ).sel(time=slice("2015-01-01", "2050-12-31"))["TREFHTMN"]

Inspecting the historical data:

In [ ]:
trefhtmn_hist

## 2.1 Dask Chunking

The `dask` package enables easy parallelism with `xarray`, but generally requires careful consideration in practice. Note that the chunking here is relatively small, ~216 KB, and segmented across time. This is not ideal for HDP calculations wich are time-dependent. Let's rechunk across *space* (HDP is spatially independent) and target larger sizes for our HPC cores to crunch through (ideally 100-300 MB).

In [ ]:
trefhtmn_hist = trefhtmn_hist.chunk(dict(time=-1, lat=38, lon=72))
trefhtmn_ssp370 = trefhtmn_ssp370.chunk(dict(time=-1, lat=38, lon=72))

trefhtmn_hist

In [ ]:
trefhtmn_ssp370

Great! The chunking no longer segments the time dimension and leverages a bit more memory for greater efficiency.

## 3. The HDP Workflow

### 3.1 Formatting Inputs

The first step is to format the data so that the HDP can standardize across units and coordinates:

In [ ]:
import hdp.measure

historical_measures = hdp.measure.format_standard_measures(
    temp_datasets=[trefhtmn_hist]
)
ssp370_measures = hdp.measure.format_standard_measures(
    temp_datasets=[trefhtmn_ssp370]
)

Notice that the units have been converted to Celsius and the metadata has changed:

In [ ]:
historical_measures

### 3.2 Calculating the Percentile Threshold

The first parameter selection to make is what thresholds should define extreme heat; what days are considered hot?

To do this, we need to select a baseline period from which we can derive a percentile-based threshold. We can define multiple percentiles to evaluate simultaneously. For this example, we will evaluate the 90th, 91st, 92nd, ... 99th percentiles from  1960 to 1990.

In [ ]:
import numpy as np
import hdp.threshold

percentiles = np.arange(0.9, 1.0, 0.01)

thresholds_dataset = hdp.threshold.compute_thresholds(
    historical_measures,
    percentiles
)

thresholds_dataset

Notice the new `doy` "day-of-year" and `percentile` dimensions.

### 3.3 Defining Heatwaves and Calculating the Metrics

The last parameter selection is our sample of heatwave definitions. Each definition is made up of three digits that correspond to different aspects of the heatwave pattern:

1. The minimum number of hot days to start a heatwave event.

2. The maximum number of non-hot days that can follow the start of a heatwave event (creating a small break).

3. The maximum number of subsequent events that can come after the break (and be considered part of the starting heatwave).

The simplest definitions allow no breaks, and only test for minimum consecutive days (here we test for 3-day, 4-day, and 5-day heatwaves):

In [ ]:
definitions = [ [3,0,0], [4,0,0], [5,0,0] ]

We can expand the sample to include 1-day breaks as part of the heatwave:

In [ ]:
definitions += [ [3,1,1], [4,1,1], [5,1,1] ]

Then, we compare our test data, the SSP3-7.0 daily minimum temperatures, against our thresholds derived from the baseline historical:

In [ ]:
import hdp.metric

metrics_dataset = hdp.metric.compute_group_metrics(ssp370_measures, thresholds_dataset, definitions)
metrics_dataset

The HDP will automatically compute four heatwave metrics for every combination of definition and percentile. This can be used to compare results across the parameter space and is particularly useful for sensitivty tests.

Notice that by default, the computation doesn't begin immediately. This is because the data is lazily loaded using Dask, which we can use to leverage parallel computing on Casper:

### 3.4 Executing the Computation in Parallel with Dask

The `dask` package can be used to connect to Casper's job queue. See [NCAR documentation on using Dask jobqueue](https://ncar.github.io/dask-tutorial/notebooks/05-dask-hpc.html) for more information.

In [ ]:
import dask
from dask_jobqueue import PBSCluster
from dask.distributed import Client

# Create a PBS cluster object
cluster = PBSCluster(
    job_name = 'dask-wk23-hpc',
    cores = 1,
    memory = '4GiB',
    processes = 1,
    resource_spec = 'select=1:ncpus=1:mem=4GB',
    queue = 'casper',
    walltime = '30:00',
    interface = 'ext',
    account = 'UCLA0088'
)

# Start up 16 new workers
cluster.scale(16)
# Create the client to load the Dashboard
client = Client(cluster)
# Wait for the workers
client.wait_for_workers(16)
client

Once the Dask cluster is online, you can execute the task graph by calling `.compute()`:

In [ ]:
%%time
metrics_dataset = metrics_dataset.compute()

## 4. Visualizing the Data

Once the data is computed, we can quickly visualize the results by using the HDP to generate a deck of figures stored in a Jupyter notebook:

In [ ]:
from hdp.graphics.notebook import create_notebook

figure_notebook = create_notebook(metrics_dataset)
figure_notebook.save_notebook(f"FigureDeck_HeatHack2026.ipynb")